# 🔵 BrailleVision - YOLOv8 Training Notebook
**BrailleVision Hackathon 2026**

This notebook trains a YOLOv8 model to detect and classify Braille cells from real physical images.

**Steps:**
1. Install dependencies
2. Connect to Roboflow dataset
3. Train YOLOv8n model
4. Evaluate and export
5. Download weights


In [ ]:
# ── Step 1: Install Dependencies ──────────────────────────────────────────────
!pip install ultralytics roboflow supervision -q

import ultralytics
ultralytics.checks()
print('✅ Ultralytics ready')

In [ ]:
# ── Step 2: Download Dataset from Roboflow ────────────────────────────────────
# Go to https://universe.roboflow.com and search 'braille detection'
# Get your API key from roboflow.com/settings/api

from roboflow import Roboflow
import os

# PASTE YOUR ROBOFLOW API KEY HERE:
RF_API_KEY = 'YOUR_ROBOFLOW_API_KEY'

rf = Roboflow(api_key=RF_API_KEY)

# Option A: Use an existing braille dataset (recommended)
# Search roboflow universe for braille datasets - pick one with 500+ images
project = rf.workspace().project('braille-detection')  # adjust workspace/project
dataset = project.version(1).download('yolov8')

print(f'✅ Dataset downloaded to: {dataset.location}')
print(f'   Classes: {dataset.classes}')

In [ ]:
# ── Step 2b: If no Roboflow dataset available - generate synthetic ─────────────
# Only run this if Step 2 failed

import cv2
import numpy as np
import os
from pathlib import Path

DOT_POSITIONS = {1:(0,0),2:(0,1),3:(0,2),4:(1,0),5:(1,1),6:(1,2)}
PATTERNS = {
    'a':[1],'b':[1,2],'c':[1,4],'d':[1,4,5],'e':[1,5],
    'f':[1,2,4],'g':[1,2,4,5],'h':[1,2,5],'i':[2,4],'j':[2,4,5],
    'k':[1,3],'l':[1,2,3],'m':[1,3,4],'n':[1,3,4,5],'o':[1,3,5],
    'p':[1,2,3,4],'q':[1,2,3,4,5],'r':[1,2,3,5],'s':[2,3,4],'t':[2,3,4,5],
    'u':[1,3,6],'v':[1,2,3,6],'w':[2,4,5,6],'x':[1,3,4,6],
    'y':[1,3,4,5,6],'z':[1,3,5,6]
}

def make_braille_img(word, size=640, augment=True):
    img = np.ones((size, size, 3), np.uint8) * np.random.randint(200, 240)
    noise = np.random.randint(-15, 15, img.shape, dtype=np.int8)
    img = np.clip(img.astype(np.int16)+noise, 0, 255).astype(np.uint8)
    
    chars = [c for c in word.lower() if c in PATTERNS]
    cw, ch = 70, 95
    start_x = np.random.randint(30, 80)
    start_y = np.random.randint(50, size//3)
    labels = []
    
    for i, char in enumerate(chars[:8]):
        x = start_x + i*(cw+20)
        if x+cw > size-20: break
        y = start_y
        dots = PATTERNS[char]
        dr = np.random.randint(7,12)
        for dn,(col,row) in DOT_POSITIONS.items():
            cx = x+20+col*35; cy = y+15+row*30
            if dn in dots:
                cv2.circle(img,(cx+1,cy+1),dr,(70,70,70),-1)
                cv2.circle(img,(cx,cy),dr,(100,100,100),-1)
            else:
                cv2.circle(img,(cx,cy),dr-2,(190,190,190),1)
        cls = list(PATTERNS.keys()).index(char)
        labels.append(f'{cls} {(x+cw/2)/size:.4f} {(y+ch/2)/size:.4f} {cw/size:.4f} {ch/size:.4f}')
    
    if augment:
        angle = np.random.uniform(-5,5)
        M = cv2.getRotationMatrix2D((size//2,size//2),angle,1)
        img = cv2.warpAffine(img,M,(size,size),borderValue=(225,225,225))
        brightness = np.random.uniform(0.8,1.2)
        img = np.clip(img.astype(np.float32)*brightness,0,255).astype(np.uint8)
    
    return img, labels

# Generate synthetic dataset
WORDS = ['hello','braille','vision','read','text','learn','help','world',
         'open','mind','hope','life','free','able','feel','touch','see',
         'hear','speak','think','bright','light','clear','smart','fast']

for split, words in [('train', WORDS*8), ('val', WORDS[:8])]:
    Path(f'data/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'data/labels/{split}').mkdir(parents=True, exist_ok=True)
    for i, word in enumerate(words):
        img, labels = make_braille_img(word, augment=(split=='train'))
        cv2.imwrite(f'data/images/{split}/{word}_{i}.jpg', img)
        with open(f'data/labels/{split}/{word}_{i}.txt','w') as f:
            f.write('\n'.join(labels))

# Write data.yaml
with open('data/data.yaml','w') as f:
    f.write(f"""path: /content/data\ntrain: images/train\nval: images/val\nnc: 26\nnames: {list(PATTERNS.keys())}\n""")

print(f'✅ Synthetic dataset generated')
print(f'   Train: {len(list(Path("data/images/train").glob("*.jpg")))} images')
print(f'   Val:   {len(list(Path("data/images/val").glob("*.jpg")))} images')

In [ ]:
# ── Step 3: Train YOLOv8 ──────────────────────────────────────────────────────
from ultralytics import YOLO
import torch

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')

# Load pretrained YOLOv8 nano (fastest, still accurate)
model = YOLO('yolov8n.pt')

# Training config - tuned for Braille detection
results = model.train(
    data='data/data.yaml',  # or dataset.location + '/data.yaml' if using Roboflow
    epochs=100,
    imgsz=640,
    batch=16,
    name='braille_yolov8n',
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.0,  # Braille is direction-sensitive
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    save=True,
    plots=True,
    verbose=True,
)

print('\n✅ Training complete!')
print(f'Best mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

In [ ]:
# ── Step 4: Evaluate Model ────────────────────────────────────────────────────
from ultralytics import YOLO

# Load best weights
model = YOLO('runs/detect/braille_yolov8n/weights/best.pt')

# Validate
metrics = model.val(data='data/data.yaml', imgsz=640, batch=16)

print('\n📊 Evaluation Metrics:')
print(f'  mAP@50:     {metrics.box.map50:.4f}')
print(f'  mAP@50-95:  {metrics.box.map:.4f}')
print(f'  Precision:  {metrics.box.mp:.4f}')
print(f'  Recall:     {metrics.box.mr:.4f}')

In [ ]:
# ── Step 5: Test Inference ────────────────────────────────────────────────────
from ultralytics import YOLO
from IPython.display import Image, display
import cv2
import numpy as np

model = YOLO('runs/detect/braille_yolov8n/weights/best.pt')

# Test on a validation image
import glob
test_imgs = glob.glob('data/images/val/*.jpg')
if test_imgs:
    result = model.predict(test_imgs[0], save=True, imgsz=640, conf=0.25)
    display(Image(filename=f'runs/detect/predict/{test_imgs[0].split("/")[-1]}'))

In [ ]:
# ── Step 6: Export Model ──────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO('runs/detect/braille_yolov8n/weights/best.pt')

# Export to ONNX (for broader compatibility)
model.export(format='onnx', imgsz=640, simplify=True)
print('✅ Exported to ONNX')

In [ ]:
# ── Step 7: Download Weights ──────────────────────────────────────────────────
from google.colab import files
import shutil, os

# Package weights and results
shutil.copy('runs/detect/braille_yolov8n/weights/best.pt', 'best.pt')
shutil.copy('runs/detect/braille_yolov8n/weights/last.pt', 'last.pt')

# Create results zip
shutil.make_archive('training_results', 'zip', 'runs/detect/braille_yolov8n')

print('Downloading best.pt...')
files.download('best.pt')

print('Downloading training results...')
files.download('training_results.zip')

print('✅ Download complete!')
print('   Place best.pt in backend/models/best.pt')